In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


In [2]:
val_binary = pd.read_csv("Extraction/val_binary.tsv", sep="\t")
test_binary = pd.read_csv("Extraction/test_binary.tsv", sep="\t")
metadata_cols = ["Entry", "Length", "Sequence"]
concept_cols = [c for c in val_binary.columns if c not in metadata_cols]

In [3]:
def calculate_f1_array(precision, recall):
    denom = precision + recall
    return np.divide(
        2 * precision * recall,
        denom,
        out=np.zeros_like(denom, dtype=float),
        where=denom > 0
    )

def compare_features_to_concepts_fast(
    A,
    binary_df,
    concept_cols,
    thresholds=(0, 0.15, 0.5, 0.6, 0.8),
):
    """
    A: normalized activations, shape [n_proteins, n_features]
    binary_df: val_binary/test_binary
    concept_cols: concept label columns

    Returns dataframe with:
    concept, feature, threshold, precision, recall, f1
    """

    A = np.asarray(A)
    Y = binary_df[concept_cols].values.astype(bool)

    n_proteins, n_features = A.shape
    n_concepts = Y.shape[1]

    positives = Y.sum(axis=0)  # [n_concepts]
    results = []

    for threshold in thresholds:
        A_bin = A > threshold  # [n_proteins, n_features]

        # Matrix multiplication gives TP:
        # Y.T: [n_concepts, n_proteins]
        # A_bin: [n_proteins, n_features]
        # tp: [n_concepts, n_features]
        tp = Y.T.astype(np.int32) @ A_bin.astype(np.int32)

        pred_pos = A_bin.sum(axis=0)  # [n_features]
        fp = pred_pos[None, :] - tp

        precision = np.divide(
            tp,
            tp + fp,
            out=np.zeros_like(tp, dtype=float),
            where=(tp + fp) > 0,
        )

        recall = np.divide(
            tp,
            positives[:, None],
            out=np.zeros_like(tp, dtype=float),
            where=positives[:, None] > 0,
        )

        f1 = calculate_f1_array(precision, recall)

        concept_idx, feature_idx = np.nonzero(tp > 0)

        df_t = pd.DataFrame({
            "concept": [concept_cols[i] for i in concept_idx],
            "feature": feature_idx,
            "threshold": threshold,
            "precision": precision[concept_idx, feature_idx],
            "recall": recall[concept_idx, feature_idx],
            "f1": f1[concept_idx, feature_idx],
            #"tp": tp[concept_idx, feature_idx],
            #"fp": fp[concept_idx, feature_idx],
            #"positive_labels": positives[concept_idx],
        })

        results.append(df_t)

    return pd.concat(results, ignore_index=True)

mean

In [4]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_8/embeddings_mean_sae_8_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_8/embeddings_mean_sae_8_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 8)
(25000, 8)
val f1: 0.06262
test f1: 0.05997
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,4,0.15,0.349875,0.961525,0.51306,0.352793,0.96229,0.516301


In [6]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_16/embeddings_mean_sae_16_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_16/embeddings_mean_sae_16_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 16)
(25000, 16)
val f1: 0.07847
test f1: 0.07577
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [7]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_32/embeddings_mean_sae_32_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_32/embeddings_mean_sae_32_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 32)
(25000, 32)
val f1: 0.0906
test f1: 0.08623
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,10,0.15,0.540083,0.761422,0.631932,0.540116,0.766198,0.633593


In [8]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_64/embeddings_mean_sae_64_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_64/embeddings_mean_sae_64_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 64)
(25000, 64)
val f1: 0.10778
test f1: 0.1009
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,34,0.15,0.461517,0.848677,0.597894,0.454025,0.842132,0.589973
1,GO:0005634,15,0.15,0.431754,0.682343,0.528866,0.433719,0.682207,0.530297


In [9]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_128/embeddings_mean_sae_128_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_128/embeddings_mean_sae_128_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 128)
(25000, 128)
val f1: 0.13061
test f1: 0.12354
Validation pairs with F1 > 0.5: 7
Those also with test F1 > 0.5: 6
Survival rate: 0.8571428571428571


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005506,73,0.80,1.000000,0.506083,0.672052,1.000000,0.489247,0.657040
1,heme,73,0.80,0.985577,0.488095,0.652866,0.967033,0.458333,0.621908
3,ig-like,64,0.50,0.473294,0.752358,0.581056,0.449635,0.749392,0.562044
4,1.14,73,0.80,0.822115,0.422222,0.557912,0.846154,0.408488,0.550984
5,GO:0020037,73,0.80,1.000000,0.382353,0.553191,1.000000,0.351351,0.520000
6,GO:0005634,79,0.15,0.371135,0.825608,0.512077,0.368875,0.819628,0.508775


In [10]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_320/embeddings_mean_sae_320_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_320/embeddings_mean_sae_320_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 320)
(25000, 320)
val f1: 0.15239
test f1: 0.14169
Validation pairs with F1 > 0.5: 12
Those also with test F1 > 0.5: 8
Survival rate: 0.6666666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,123,0.60,0.560976,0.779661,0.652482,0.518987,0.788462,0.625954
3,GO:0005506,53,0.80,0.958115,0.445255,0.607973,0.941860,0.435484,0.595588
2,ig-like,273,0.60,0.659574,0.584906,0.620000,0.574307,0.554745,0.564356
4,heme,53,0.80,0.942408,0.428571,0.589198,0.906977,0.406250,0.561151
1,GO:0003925,304,0.60,0.538961,0.757991,0.629981,0.466667,0.692308,0.557522
8,krab,188,0.60,0.475410,0.557692,0.513274,0.522388,0.573770,0.546875
6,transmembrane,269,0.15,0.423343,0.746994,0.540416,0.416667,0.730202,0.530577
10,GO:0005634,293,0.15,0.497168,0.512981,0.504950,0.504978,0.510126,0.507539


In [11]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_2560/embeddings_mean_sae_2560_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_2560/embeddings_mean_sae_2560_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 2560)
(25000, 2560)
val f1: 0.19129
test f1: 0.17877
Validation pairs with F1 > 0.5: 20
Those also with test F1 > 0.5: 18
Survival rate: 0.9


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,2406,0.80,0.888889,0.813559,0.849558,0.792453,0.807692,0.800000
2,c-type lectin,991,0.50,0.670588,0.662791,0.666667,0.670330,0.734940,0.701149
1,nudix hydrolase,1594,0.60,0.842105,0.581818,0.688172,0.906250,0.568627,0.698795
8,krab,717,0.50,0.527027,0.750000,0.619048,0.605263,0.754098,0.671533
3,GO:0005506,1114,0.80,1.000000,0.496350,0.663415,0.983333,0.475806,0.641304
4,ig-like,2083,0.50,0.813380,0.544811,0.652542,0.783688,0.537713,0.637807
6,heme,1114,0.80,0.985294,0.478571,0.644231,0.950000,0.445312,0.606383
7,6.2,2056,0.80,0.833333,0.512821,0.634921,0.861111,0.455882,0.596154
9,transmembrane,1100,0.15,0.520700,0.699931,0.597157,0.517026,0.694892,0.592907
5,GO:0003925,1094,0.80,0.539394,0.812785,0.648452,0.446541,0.780220,0.568000


In [12]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_5120/embeddings_mean_sae_5120_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_5120/embeddings_mean_sae_5120_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 5120)
(25000, 5120)
val f1: 0.19942
test f1: 0.18245
Validation pairs with F1 > 0.5: 24
Those also with test F1 > 0.5: 21
Survival rate: 0.875


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,4713,0.60,0.962264,0.864407,0.910714,0.897959,0.846154,0.871287
1,nudix hydrolase,1160,0.50,0.928571,0.709091,0.804124,0.942857,0.647059,0.767442
2,c-type lectin,1823,0.60,0.775000,0.720930,0.746988,0.687500,0.795181,0.737430
4,protein kinase,3883,0.50,0.721274,0.627723,0.671255,0.694882,0.644749,0.668877
3,GO:0003925,4557,0.80,0.702222,0.721461,0.711712,0.628866,0.670330,0.648936
5,GO:0005506,987,0.60,0.873950,0.506083,0.640986,0.814159,0.494624,0.615385
8,6.2,1035,0.80,0.719298,0.525641,0.607407,0.692308,0.529412,0.600000
9,transmembrane,1229,0.15,0.486625,0.768636,0.595952,0.480170,0.757456,0.587750
7,ig-like,4438,0.60,0.581897,0.636792,0.608108,0.570115,0.603406,0.586288
6,heme,987,0.60,0.861345,0.488095,0.623100,0.787611,0.463542,0.583607


min

In [13]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_8/embeddings_min_sae_8_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_8/embeddings_min_sae_8_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 8)
(25000, 8)
val f1: 0.07869
test f1: 0.07482
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 4
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,3,0.8,0.654562,0.731683,0.690977,0.695459,0.713242,0.704238
1,GO:0106310,3,0.8,0.534987,0.751244,0.624935,0.566340,0.749117,0.645030
2,GO:0004674,3,0.8,0.449070,0.680537,0.541089,0.483526,0.677057,0.564156
3,2.7,3,0.8,0.650133,0.450031,0.531884,0.697240,0.459238,0.553748


In [14]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_16/embeddings_min_sae_16_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_16/embeddings_min_sae_16_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 16)
(25000, 16)
val f1: 0.09964
test f1: 0.0944
Validation pairs with F1 > 0.5: 6
Those also with test F1 > 0.5: 5
Survival rate: 0.8333333333333334


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,11,0.8,0.714625,0.595050,0.649379,0.743041,0.633790,0.684081
3,GO:0106310,11,0.8,0.536266,0.560945,0.548328,0.553533,0.608952,0.579921
1,GO:0003924,7,0.8,0.641509,0.558394,0.597073,0.636782,0.528626,0.577685
4,GO:0005525,7,0.8,0.702306,0.445479,0.545159,0.694253,0.430813,0.531690
2,GO:0003925,7,0.8,0.410901,0.894977,0.563218,0.356322,0.851648,0.502431


In [15]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_32/embeddings_min_sae_32_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_32/embeddings_min_sae_32_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 32)
(25000, 32)
val f1: 0.10908
test f1: 0.10171
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 5
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,3,0.8,0.797680,0.612871,0.693169,0.831902,0.619178,0.709948
1,GO:0106310,3,0.8,0.627577,0.605721,0.616456,0.655215,0.628975,0.641827
2,abc transporter,21,0.8,0.430108,0.714286,0.536913,0.546512,0.580247,0.562874
3,GO:0004674,3,0.8,0.511598,0.532886,0.522025,0.555828,0.564838,0.560297
4,2.7,3,0.8,0.784794,0.373391,0.506024,0.828221,0.395894,0.535714


In [16]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_64/embeddings_min_sae_64_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_64/embeddings_min_sae_64_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 64)
(25000, 64)
val f1: 0.12738
test f1: 0.12059
Validation pairs with F1 > 0.5: 8
Those also with test F1 > 0.5: 8
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,0,0.6,0.777293,0.649635,0.707753,0.772210,0.646947,0.704050
1,protein kinase,35,0.6,0.534444,0.952475,0.684698,0.543942,0.955251,0.693174
2,GO:0005525,0,0.6,0.849345,0.517287,0.642975,0.849658,0.532097,0.654386
5,GO:0106310,9,0.6,0.419463,0.932836,0.578704,0.426616,0.917550,0.582430
3,ig-like,11,0.6,0.618812,0.589623,0.603865,0.572464,0.576642,0.574545
6,2.7,35,0.6,0.525556,0.580012,0.551443,0.525221,0.592375,0.556781
7,GO:0004674,35,0.6,0.372222,0.899329,0.526523,0.379615,0.910224,0.535780
4,GO:0003925,0,0.6,0.434498,0.908676,0.587888,0.373576,0.901099,0.528180


In [17]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_128/embeddings_min_sae_128_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_128/embeddings_min_sae_128_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 128)
(25000, 128)
val f1: 0.14476
test f1: 0.13533
Validation pairs with F1 > 0.5: 8
Those also with test F1 > 0.5: 8
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
2,protein kinase,46,0.6,0.530354,0.787129,0.633719,0.549430,0.791781,0.648709
0,GO:0003925,110,0.8,0.699482,0.616438,0.655340,0.666667,0.582418,0.621701
1,krab,90,0.8,0.565217,0.750000,0.644628,0.557143,0.639344,0.595420
3,nudix hydrolase,24,0.8,0.750000,0.545455,0.631579,0.675676,0.490196,0.568182
4,GO:0003924,110,0.6,0.527778,0.728102,0.611963,0.468593,0.711832,0.565152
7,GO:0106310,46,0.6,0.409606,0.763682,0.533218,0.417617,0.776207,0.543057
5,GO:0005525,110,0.6,0.558201,0.561170,0.559682,0.502513,0.570613,0.534402
6,ig-like,64,0.6,0.396896,0.844340,0.539970,0.370246,0.805353,0.507280


In [18]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_320/embeddings_min_sae_320_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_320/embeddings_min_sae_320_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 320)
(25000, 320)
val f1: 0.14645
test f1: 0.13401
Validation pairs with F1 > 0.5: 7
Those also with test F1 > 0.5: 7
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,33,0.6,0.726354,0.783168,0.753692,0.746383,0.800913,0.772687
1,GO:0106310,33,0.6,0.556474,0.753731,0.640254,0.556596,0.770318,0.646245
2,krab,245,0.8,0.472527,0.826923,0.601399,0.511364,0.737705,0.604027
4,GO:0004674,33,0.6,0.486685,0.711409,0.577972,0.508085,0.744389,0.603945
5,2.7,33,0.6,0.708907,0.473329,0.567647,0.719149,0.495601,0.586806
6,GO:0004252,71,0.6,0.628099,0.515254,0.566108,0.551095,0.493464,0.520690
3,c-type lectin,141,0.6,0.468531,0.779070,0.585153,0.397436,0.746988,0.518828


In [19]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_2560/embeddings_min_sae_2560_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_2560/embeddings_min_sae_2560_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 2560)
(25000, 2560)
val f1: 0.17134
test f1: 0.15631
Validation pairs with F1 > 0.5: 17
Those also with test F1 > 0.5: 10
Survival rate: 0.5882352941176471


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,1561,0.6,0.715361,0.940594,0.812660,0.726308,0.937900,0.818653
1,GO:0106310,1412,0.8,0.906977,0.582090,0.709091,0.910112,0.572438,0.702820
2,abc transporter,602,0.8,0.600000,0.750000,0.666667,0.712329,0.641975,0.675325
5,GO:0004674,1561,0.6,0.493223,0.879195,0.631934,0.513437,0.905237,0.655235
6,2.7,1561,0.6,0.701807,0.571429,0.629943,0.702263,0.582405,0.636743
4,helicase,643,0.8,0.662722,0.622222,0.641834,0.619048,0.611765,0.615385
7,if rod,1149,0.8,0.606557,0.506849,0.552239,0.569444,0.554054,0.561644
8,GO:0005524,1510,0.6,0.603391,0.505895,0.550358,0.616796,0.502324,0.553705
15,rrm,506,0.6,0.422785,0.670683,0.518634,0.420048,0.674330,0.517647
3,6.2,2487,0.8,0.785714,0.564103,0.656716,0.651163,0.411765,0.504505


In [20]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_5120/embeddings_min_sae_5120_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_5120/embeddings_min_sae_5120_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 5120)
(25000, 5120)
val f1: 0.17113
test f1: 0.15497
Validation pairs with F1 > 0.5: 11
Those also with test F1 > 0.5: 8
Survival rate: 0.7272727272727273


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,4769,0.6,0.744000,0.920792,0.823009,0.777607,0.926027,0.845352
2,GO:0106310,4769,0.6,0.593600,0.922886,0.722493,0.606595,0.931684,0.734789
1,6.2,2181,0.8,0.862069,0.641026,0.735294,0.775862,0.661765,0.714286
3,GO:0004674,4769,0.6,0.518400,0.869799,0.649624,0.542945,0.882793,0.672365
4,2.7,4769,0.6,0.720000,0.551809,0.624783,0.746166,0.570674,0.646726
8,GO:0005524,4089,0.6,0.650833,0.460818,0.539586,0.657534,0.462151,0.542796
7,GO:0003925,5051,0.8,0.644970,0.497717,0.561856,0.627737,0.472527,0.539185
5,GO:0004252,2444,0.6,0.800000,0.433898,0.562637,0.720000,0.411765,0.523909
